In [1]:
# Install required packages
!pip install -q anthropic google-generativeai requests

In [1]:
import os
import re
from datetime import datetime

# ========================================
# CHOOSE YOUR PROVIDER AND PASTE API KEY
# ========================================
PROVIDER = "gemini"  # Options: "claude" or "gemini"

CLAUDE_API_KEY = "YOUR_CLAUDE_KEY_HERE"  # Get from: https://console.anthropic.com/
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]  # Get from: https://aistudio.google.com/app/apikey

# ========================================

# Setup based on provider
if PROVIDER == "claude":
    import anthropic
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    print("✅ Using Claude (Anthropic)")
    print("   Model: claude-3-haiku-20240307")
    
elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    client = genai.GenerativeModel('models/gemini-2.5-flash')
    print("✅ Using Gemini")
    print("   ⚠️  Note: Only 20 requests/day on free tier")


✅ Using Gemini
   ⚠️  Note: Only 20 requests/day on free tier


c:\Users\jettg\miniconda3\envs\ece285\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\jettg\AppData\Local\Temp\ipykernel_26888\3570226072.py:23: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


# Baseline Prompt

In [ ]:
import pandas as pd

def call_llm(df, Turb_ID, base_day):
    turbine_data = df[df['TurbID'] == Turb_ID]
    # Drop rows where 'Wspd' (Wind Speed) is missing
    turbine_data = turbine_data.dropna(subset=['Wspd'])

    day_data = turbine_data[turbine_data['Day'] == base_day].drop(columns=['Day', 'TurbID'])

    data_str = day_data.to_csv(index=False, sep=',')

    prompt = f"""
Context: You are a energy forecaster for wind turbines. You have been provided with Wind Turbine {Turb_ID}'s sensor data for Day {base_day}. 
The data is sampled every 10 minutes (144 rows total).

Columns: Tmstamp (time stamp), Wspd (wind speed in m/s), Wdir (The angle between the wind direction and the position of turbine nacelle in degrees), Etmp (Temperature of the surounding environment in celcius), Itmp (Temperature inside the turbine nacelle in celcius), Ndir (Nacelle direction, i.e., the yaw angle of the nacelle in degrees), Pab1 (Pitch angle of blade 1 in degrees), Pab2 (Pitch angle of blade 2 in degrees), Pab3 (Pitch angle of blade 3 in degrees), Prtv (Reactive power in kW), Patv (Active power in kW)

Input Data:
{data_str}

Task: Predict the 'Patv' (Active Power) for the next 24 hours (Day {base_day + 1}). Do not write code. Analyze the relationship between wind speed, temperature, and power in the provided data, and output your best estimate for the next 24 hours based on these trends. Return only a list of 144 numerical values separated by commas.
"""
    
    if PROVIDER == "claude":
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    else:  # gemini
        response = client.generate_content(prompt)
        return response.text

## Simple prompting is not ENOUGH

# Prompt with provided context (Future wind conditions)

In [ ]:
import pandas as pd

def call_llm(df, Turb_ID, base_day):
    turbine_data = df[df['TurbID'] == Turb_ID]
    # Drop rows where 'Wspd' (Wind Speed) is missing
    turbine_data = turbine_data.dropna(subset=['Wspd'])

    day_data = turbine_data[turbine_data['Day'] == base_day].drop(columns=['Day', 'TurbID'])

    data_str = day_data.to_csv(index=False, sep=',')

    forecast_data = (turbine_data[(turbine_data['Day'] == base_day + 1)].head(36)[['Tmstamp', 'Wspd', 'Wdir', 'Etmp']])

    forecast_str = forecast_data.to_csv(index=False)

    prompt = f"""
Context: You are a energy forecaster for wind turbines. 

You are given:
1) Historical data for Wind Turbine {Turb_ID}'s sensor data for Day {base_day}. 
2) Forecasted wind conditions for the first 6 hours of Day {base_day + 1}

The columns of the historical data are: Tmstamp (time stamp), Wspd (wind speed in m/s), Wdir (The angle between the wind direction and the position of turbine nacelle in degrees), Etmp (Temperature of the surounding environment in celcius), Itmp (Temperature inside the turbine nacelle in celcius), Ndir (Nacelle direction, i.e., the yaw angle of the nacelle in degrees), Pab1 (Pitch angle of blade 1 in degrees), Pab2 (Pitch angle of blade 2 in degrees), Pab3 (Pitch angle of blade 3 in degrees), Prtv (Reactive power in kW), Patv (Active power in kW)

Input Data:
1) Historical data (144 rows, 10-min intervals): 
{data_str}

2) Forecasted Conditions (Next 6 hours):
{forecast_str}

Task: Predict the 'Patv' (Active power in kW) for the next 6 hours (Day {base_day + 1}). Using the historical data, analyze the relationship between wind speed, temperature, and power in the provided data. Then use the provided forecasted wind conditions to predict Patv for the next 36 time steps (6 hours) based on these trends. Return only a list of 36 numerical values separated by commas. Do not write code.
"""
    
    if PROVIDER == "claude":
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    else:  # gemini
        response = client.generate_content(prompt)
        return response.text

In [16]:
df = pd.read_csv('wtbdata_245days.csv')
Turb_ID = 1
base_day = 2
output = call_llm(df, Turb_ID, base_day)
print(output)

1545,1518,1468,1550,1463,1517,1545,1550,1545,1550,1518,1533,1142,512,690,909,826,1008,1119,991,995,1002,1257,1550,1545,1463,1540,1531,1451,1503,1550,1550,1545,1538,1522,1506


In [17]:
target_day = base_day + 1
actual_data = df[(df['TurbID'] == Turb_ID) & (df['Day'] == target_day)]

actual_values = actual_data['Patv'].dropna().tolist()
predicted_values = [float(val.strip()) for val in output.split(',')]

In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Ensure both lists are the same length
min_len = min(len(actual_values), len(predicted_values))
y_true = actual_values[:min_len]
y_pred = predicted_values[:min_len]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Mean Absolute Error: {mae:.2f} kW")
print(f"Root Mean Square Error: {rmse:.2f} kW")

Mean Absolute Error: 24.40 kW
Root Mean Square Error: 37.24 kW


144 144
